# 13B — Pairwise SAR spatial transitions with common valid support

**Priority 2.** Generate aligned rasters for matched same-orbit acquisitions, compute four mutually exclusive threshold-transition classes and continuous ΔVV/ΔVH. Missing in either scene is **not** dry. Requires Earth Engine for scene rasters.

**Execution:** This is a source workflow, NOT results. A successful extract needs your own data / Earth Engine session. Do not treat low-VV detections as validated water, flood depth or methane flux.

In [1]:
# Every notebook uses the same ROOT; no hard-coded drive or EE project ID.
from pathlib import Path
import os, sys
starting = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [starting, *starting.parents]:
    if (candidate/"src/analysis13/core.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Extract the ZIP into the project root and open Jupyter there, or set HAKALUKI_ROOT")
sys.path.insert(0, str(ROOT))
print("Research root:", ROOT)


Research root: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project


## Choose reproducible event pairs

Start with one suspicious pair, inspect outputs. Set `PAIR_NAMES=None` after QA to generate all six. Raw per-scene downloads are cached outside Git.

In [2]:
from src.analysis13.pairs import PAIRS, select_pairs
from src.analysis13.core import load_scenes

scenes = load_scenes(ROOT)

# Explicitly select all six predefined scene pairs
PAIR_NAMES = [pair.name for pair in PAIRS]

selected_pairs = select_pairs(scenes, names=PAIR_NAMES)

print("Requested pairs:", PAIR_NAMES)
print("Selected pairs:", len(selected_pairs))

for pair, before, after in selected_pairs:
    print(pair.name, "|", before[:38], "->", after[:38])

assert len(selected_pairs) == 6, (
    f"Expected 6 pairs, but found {len(selected_pairs)}"
)

Requested pairs: ['2022_may_expansion', '2022_june_stable', '2023_june_expansion', '2024_june_early_to_mid', '2024_june_mid_to_late', '2025_may_june_descending']
Selected pairs: 6
2022_may_expansion | S1A_IW_GRDH_1SDV_20220511T115642_20220 -> S1A_IW_GRDH_1SDV_20220523T115643_20220
2022_june_stable | S1A_IW_GRDH_1SDV_20220604T115644_20220 -> S1A_IW_GRDH_1SDV_20220628T115646_20220
2023_june_expansion | S1A_IW_GRDH_1SDV_20230611T115649_20230 -> S1A_IW_GRDH_1SDV_20230623T115650_20230
2024_june_early_to_mid | S1A_IW_GRDH_1SDV_20240605T115652_20240 -> S1A_IW_GRDH_1SDV_20240617T115651_20240
2024_june_mid_to_late | S1A_IW_GRDH_1SDV_20240617T115651_20240 -> S1A_IW_GRDH_1SDV_20240629T115651_20240
2025_may_june_descending | S1A_IW_GRDH_1SDV_20250521T234742_20250 -> S1A_IW_GRDH_1SDV_20250602T234742_20250


## Run aligned per-scene spatial processing

Produces comparative maps and three-threshold transition table. May fail on EE download size or permissions; keep and share exact traceback to fix.

In [3]:
from src.analysis13.spatial import run_spatial
RUN_SPATIAL_EE = True
if RUN_SPATIAL_EE:
    result = run_spatial(ROOT, pair_names=PAIR_NAMES, force=False)
    print(result)
else:
    print('Spatial EE downloads intentionally deferred')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Spatial pair 1/6: 2022_may_expansion


F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:129: RuntimeWarning: invalid value encountered in subtract
  maps={"joint_valid":good,"vv_delta":np.where(good,late[0]-early[0],np.nan),
F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:130: RuntimeWarning: invalid value encountered in subtract
  "vh_delta":np.where(good & (early[1]>-100) & (late[1]>-100),late[1]-early[1],np.nan)}


Spatial pair 2/6: 2022_june_stable


F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:129: RuntimeWarning: invalid value encountered in subtract
  maps={"joint_valid":good,"vv_delta":np.where(good,late[0]-early[0],np.nan),
F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:130: RuntimeWarning: invalid value encountered in subtract
  "vh_delta":np.where(good & (early[1]>-100) & (late[1]>-100),late[1]-early[1],np.nan)}


...

Spatial pair 5/6: 2024_june_mid_to_late


F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:129: RuntimeWarning: invalid value encountered in subtract
  maps={"joint_valid":good,"vv_delta":np.where(good,late[0]-early[0],np.nan),
F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:130: RuntimeWarning: invalid value encountered in subtract
  "vh_delta":np.where(good & (early[1]>-100) & (late[1]>-100),late[1]-early[1],np.nan)}


Spatial pair 6/6: 2025_may_june_descending


F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:129: RuntimeWarning: invalid value encountered in subtract
  maps={"joint_valid":good,"vv_delta":np.where(good,late[0]-early[0],np.nan),
F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\src\analysis13\spatial.py:130: RuntimeWarning: invalid value encountered in subtract
  "vh_delta":np.where(good & (early[1]>-100) & (late[1]>-100),late[1]-early[1],np.nan)}


{'created_utc': '2026-10-14T18:32:29.644461+00:00', 'status': 'executed_unvalidated_SAR_screening_change', 'pair_count': 6, 'grid_crs': 'EPSG:32646', 'grid_cell_m': 30, 'grid_pixels': 330038, 'area_support': 'common observed cell centres, not exact polygon-boundary EE pixelArea', 'spatial_map_files': {'2022_may_expansion': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\results\\figures\\analysis13\\13B_2022_may_expansion_spatial_change.png', '2022_june_stable': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\results\\figures\\analysis13\\13B_2022_june_stable_spatial_change.png', '2023_june_expansion': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\results\\figures\\analysis13\\13B_2023_june_expansion_spatial_change.png', '2024_june_early_to_mid': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\results\\figures\\analysis13\\13B_2024_june_early_to_mid_spatial_change.png', '2024_june_mid_to_late': 'F:\\Wetland T

## Inspect map and transition area definitions

Grid-cell centre area may differ slightly from Earth Engine pixel-area measurement; report shared grid support and each class. A threshold crossing is not validated inundation.

In [4]:
from pathlib import Path
import pandas as pd
p=ROOT/'results/tables/analysis13/13B_paired_spatial_transitions.csv'
if p.exists():
    display(pd.read_csv(p))
else:
    print('Pending Earth Engine per-pair processing')

,pair,early_scene_id,late_scene_id,threshold_db,shared_grid_pixels,shared_grid_support_km2,neither_km2,both_km2,newly_low_vv_km2,no_longer_low_vv_km2,net_km2,changed_km2,median_VV_change_db,p10_VV_change_db,p90_VV_change_db
0,2022_may_expansion,S1A_IW_GRDH_1SDV_20220511T115642_20220511T1157...,S1A_IW_GRDH_1SDV_20220523T115643_20220523T1157...,-19.0,191413,172.2717,22.9545,25.9317,121.4019,1.9836,119.4183,123.3855,-10.171384,-14.565395,-0.675176
1,2022_may_expansion,S1A_IW_GRDH_1SDV_20220511T115642_20220511T1157...,S1A_IW_GRDH_1SDV_20220523T115643_20220523T1157...,-17.0,191413,172.2717,9.6309,40.4154,121.7421,0.4833,121.2588,122.2254,-10.171384,-14.565395,-0.675176
2,2022_may_expansion,S1A_IW_GRDH_1SDV_20220511T115642_20220511T1157...,S1A_IW_GRDH_1SDV_20220523T115643_20220523T1157...,-15.0,191413,172.2717,7.1676,47.6397,117.1620,0.3024,116.8596,117.4644,-10.171384,-14.565395,-0.675176
3,2022_june_stable,S1A_IW_GRDH_1SDV_20220604T115644_20220604T1157...,S1A_IW_GRDH_1SDV_20220628T115646_20220628T1157...,-19.0,191413,172.2717,10.6434,122.7555,27.6849,11.1879,16.4970,38.8728,-0.991331,-4.478335,1.688739
4,2022_june_stable,S1A_IW_GRDH_1SDV_20220604T115644_20220604T1157...,S1A_IW_GRDH_1SDV_20220628T115646_20220628T1157...,-17.0,191413,172.2717,5.4243,158.3631,6.4323,2.0520,4.3803,8.4843,-0.991331,-4.478335,1.688739
5,2022_june_stable,S1A_IW_GRDH_1SDV_20220604T115644_20220604T1157...,S1A_IW_GRDH_1SDV_20220628T115646_20220628T1157...,-15.0,191413,172.2717,3.9654,161.9352,5.1390,1.2321,3.9069,6.3711,-0.991331,-4.478335,1.688739
6,2023_june_expansion,S1A_IW_GRDH_1SDV_20230611T115649_20230611T1157...,S1A_IW_GRDH_1SDV_20230623T115650_20230623T1157...,-19.0,191413,172.2717,24.1218,25.3746,122.3433,0.4320,121.9113,122.7753,-10.126246,-14.768630,-0.843874
7,2023_june_expansion,S1A_IW_GRDH_1SDV_20230611T115649_20230611T1157...,S1A_IW_GRDH_1SDV_20230623T115650_20230623T1157...,-17.0,191413,172.2717,14.3946,36.7146,121.1130,0.0495,121.0635,121.1625,-10.126246,-14.768630,-0.843874
8,2023_june_expansion,S1A_IW_GRDH_1SDV_20230611T115649_20230611T1157...,S1A_IW_GRDH_1SDV_20230623T115650_20230623T1157...,-15.0,191413,172.2717,11.4894,42.3342,118.3887,0.0594,118.3293,118.4481,-10.126246,-14.768630,-0.843874
9,2024_june_early_to_mid,S1A_IW_GRDH_1SDV_20240605T115652_20240605T1157...,S1A_IW_GRDH_1SDV_20240617T115651_20240617T1157...,-19.0,191413,172.2717,10.9341,54.6489,1.3860,105.3027,-103.9167,106.6887,4.504417,1.229820,7.799074


In [5]:
import json
import numpy as np
import pandas as pd

tables = ROOT / "results/tables/analysis13"
figures = ROOT / "results/figures/analysis13"

df = pd.read_csv(tables / "13B_paired_spatial_transitions.csv")

with open(tables / "13B_manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

assert manifest["pair_count"] == 6
assert len(df) == 18
assert df["pair"].nunique() == 6

numeric_cols = [
    "shared_grid_support_km2",
    "neither_km2",
    "both_km2",
    "newly_low_vv_km2",
    "no_longer_low_vv_km2",
    "net_km2",
    "changed_km2",
    "median_VV_change_db",
    "p10_VV_change_db",
    "p90_VV_change_db",
]

assert np.isfinite(df[numeric_cols].to_numpy()).all(), \
    "Nonfinite values detected in the spatial statistics"

assert all(
    (figures / f"13B_{name}_spatial_change.png").exists()
    for name in df["pair"].unique()
)

print("PASS: Six pairs, 18 records, finite statistics and six maps")

display(
    df[df["threshold_db"] == -17][
        [
            "pair",
            "shared_grid_support_km2",
            "newly_low_vv_km2",
            "no_longer_low_vv_km2",
            "net_km2",
            "median_VV_change_db",
        ]
    ]
)

PASS: Six pairs, 18 records, finite statistics and six maps


,pair,shared_grid_support_km2,newly_low_vv_km2,no_longer_low_vv_km2,net_km2,median_VV_change_db
1,2022_may_expansion,172.2717,121.7421,0.4833,121.2588,-10.171384
4,2022_june_stable,172.2717,6.4323,2.0520,4.3803,-0.991331
7,2023_june_expansion,172.2717,121.1130,0.0495,121.0635,-10.126246
10,2024_june_early_to_mid,172.2717,2.0538,20.6334,-18.5796,4.504417
13,2024_june_mid_to_late,172.2717,21.3129,0.7596,20.5533,-1.998861
16,2025_may_june_descending,172.2717,64.1151,22.0464,42.0687,-1.811575
